# Tutorial 13: Best Practices and Advanced Topics

In this tutorial, we'll cover best practices and advanced topics for developing and deploying LangChain and LangGraph applications. We'll explore performance optimization, handling rate limits and API costs, security considerations, deployment strategies, and monitoring and logging in production.

## Setup

First, let's import the necessary libraries and set up our environment:

In [1]:
import os
import asyncio
import time
from typing import List, Dict, Any
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field, field_validator
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Initialize Groq LLM
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 1. Performance optimization techniques

Let's explore some performance optimization techniques for LangChain applications:

In [2]:
# LCEL chain: prompt | llm | parser (replaces deprecated LLMChain)
prompt = PromptTemplate.from_template("Write a short paragraph about {topic}.")
chain = prompt | llm | StrOutputParser()

# Sequential processing
def process_sequential(topics: List[str]) -> List[str]:
    return [chain.invoke({"topic": t}) for t in topics]

# Async processing — all requests launch concurrently
async def process_async(topics: List[str]) -> List[str]:
    tasks = [chain.ainvoke({"topic": t}) for t in topics]
    return await asyncio.gather(*tasks)

topics = ["Python", "Machine Learning", "Data Science"]

t0 = time.time()
seq_results = process_sequential(topics)
t_seq = time.time() - t0
print(f"Sequential: {t_seq:.2f}s | {len(seq_results)} results")

t0 = time.time()
async_results = await process_async(topics)
t_async = time.time() - t0
print(f"Async:      {t_async:.2f}s | {len(async_results)} results")

if t_async > 0:
    print(f"Speedup: {t_seq / t_async:.1f}x")

Sequential: 0.85s | 3 results
Async:      0.43s | 3 results
Speedup: 2.0x


## 2. Handling rate limits and API costs

Let's implement a system to handle rate limits and track API costs:

In [3]:
# Groq exposes token usage in response_metadata — no OpenAI callback needed
def run_with_token_tracking(prompt: str) -> Dict[str, Any]:
    response = llm.invoke([HumanMessage(content=prompt)])
    usage = response.response_metadata.get("token_usage", {})
    return {
        "response": response.content,
        "total_tokens": usage.get("total_tokens", 0),
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "total_time": usage.get("total_time", 0),
    }

result = run_with_token_tracking("Explain machine learning in one paragraph.")
print(f"Response: {result['response'][:200]}")
print(f"Prompt tokens:     {result['prompt_tokens']}")
print(f"Completion tokens: {result['completion_tokens']}")
print(f"Total tokens:      {result['total_tokens']}")
print(f"Total time (s):    {result['total_time']:.3f}")

Response: Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data and make predictions or decisions based on that data. It's a way for computers to analyze p
Prompt tokens:     43
Completion tokens: 128
Total tokens:      171
Total time (s):    0.196


## 3. Security considerations

When working with LangChain and LangGraph applications, consider the following security best practices:

1. Use environment variables for API keys and sensitive information
2. Implement input validation and sanitization
3. Use HTTPS for all API communication
4. Implement proper authentication and authorization
5. Regularly update dependencies
6. Be cautious with user-provided content in prompts

Here's an example of input validation:

In [4]:
from pydantic import BaseModel, Field, field_validator

class UserInput(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=1000)
    
    @field_validator('prompt')
    def no_sensitive_info(cls, v):
        sensitive_words = ['password', 'credit card', 'social security']
        if any(word in v.lower() for word in sensitive_words):
            raise ValueError("Input contains sensitive information")
        return v

# Example usage
try:
    user_input = UserInput(prompt="Tell me about AI")
    print("Valid input:", user_input)
    
    invalid_input = UserInput(prompt="My password is 123456")
except ValueError as e:
    print("Invalid input:", str(e))

Valid input: prompt='Tell me about AI'
Invalid input: 1 validation error for UserInput
prompt
  Value error, Input contains sensitive information [type=value_error, input_value='My password is 123456', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## 4. Deploying LangChain and LangGraph applications

Let's create a simple FastAPI application that uses our LangChain model:

In [5]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

web_app = FastAPI()

class Query(BaseModel):
    text: str

@web_app.post("/generate")
async def generate_text(query: Query):
    try:
        result = await chain.ainvoke({"topic": query.text})
        return {"generated_text": result}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# To run: uvicorn module_name:web_app --reload
print("FastAPI app defined.")
print("Endpoints:", [r.path for r in web_app.routes])

FastAPI app defined.
Endpoints: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/generate']


## 5. Monitoring and logging in production

Let's implement basic monitoring using Prometheus metrics:

In [6]:
from prometheus_client import Counter, Histogram

REQUESTS = Counter('api_requests_total', 'Total API requests')
LATENCY = Histogram('api_latency_seconds', 'API latency in seconds')

@web_app.post("/generate-monitored")
async def generate_monitored(query: Query):
    REQUESTS.inc()
    with LATENCY.time():
        try:
            result = await chain.ainvoke({"topic": query.text})
            return {"generated_text": result}
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

# Metrics are exposed at GET /metrics by adding prometheus_fastapi_instrumentator
print(f"REQUESTS metric:  {REQUESTS._name}")
print(f"LATENCY metric:   {LATENCY._name}")
print("Metrics endpoint: GET /metrics (via prometheus_fastapi_instrumentator)")

REQUESTS metric:  api_requests
LATENCY metric:   api_latency_seconds
Metrics endpoint: GET /metrics (via prometheus_fastapi_instrumentator)


## Conclusion

In this tutorial, we've covered several best practices and advanced topics for LangChain and LangGraph applications:

1. Performance optimization techniques, including caching and asynchronous processing
2. Handling rate limits and tracking API costs
3. Security considerations and input validation
4. Deploying applications using FastAPI
5. Monitoring and logging in production using Prometheus metrics

These practices will help you build more efficient, secure, and production-ready AI applications using LangChain and LangGraph.

## Next Steps

To further improve your LangChain and LangGraph applications:

1. Implement more advanced caching strategies, such as using Redis for distributed caching
2. Explore containerization (e.g., Docker) for easier deployment and scaling
3. Implement more comprehensive logging and error handling
4. Set up CI/CD pipelines for automated testing and deployment
5. Explore advanced monitoring and alerting systems
6. Consider using a reverse proxy (e.g., Nginx) for load balancing and additional security
7. Implement rate limiting and request throttling to protect your API

Remember to always follow best practices for security, performance, and scalability as you develop and deploy your AI applications